# 02 — Data Quality & Validation

## Project: System Capacity & Care Load Analytics for Unaccompanied Children

### Step 2A — Load Structured Dataset and Confirm Input Readiness

This notebook follows the project guideline section: **Data Quality & Validation**.

The goal of this step is to:
- Load the structured dataset created in Notebook 01
- Confirm file path readiness
- Check dataset shape
- Review columns
- Review current data types
- Confirm the dataset is ready for validation

No column renaming, numeric conversion, logical validation, anomaly flagging, KPI calculation, or visualization is performed in this step.

In [1]:
# ============================================================
# Step 2A — Load Structured Dataset and Confirm Input Readiness
# Project: System Capacity & Care Load Analytics for Unaccompanied Children
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings for better notebook readability
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

print("Step 2A — Load Structured Dataset and Confirm Input Readiness")
print("=" * 75)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STRUCTURED_FILE_PATH = DATA_CLEANED_DIR / "01_structured_uac_data.csv"

print("Project paths configured successfully.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Structured input file: {STRUCTURED_FILE_PATH}")

# ------------------------------------------------------------
# 2. Check if structured file exists
# ------------------------------------------------------------

file_exists = STRUCTURED_FILE_PATH.exists()

print(f"\nStructured input file exists: {file_exists}")

if not file_exists:
    raise FileNotFoundError(
        f"Structured dataset not found at: {STRUCTURED_FILE_PATH}. "
        "Please complete Notebook 01 Step 1E before running Notebook 02."
    )

# ------------------------------------------------------------
# 3. Load structured dataset
# ------------------------------------------------------------

df_hhs_quality = pd.read_csv(STRUCTURED_FILE_PATH)

print("\nStructured dataset loaded successfully.")
print(f"Dataset shape: {df_hhs_quality.shape[0]:,} rows × {df_hhs_quality.shape[1]:,} columns")

# ------------------------------------------------------------
# 4. Column review
# ------------------------------------------------------------

print("\nColumn names:")
for idx, col in enumerate(df_hhs_quality.columns, start=1):
    print(f"{idx}. {col}")

# ------------------------------------------------------------
# 5. Dataset preview
# ------------------------------------------------------------

print("\nDataset preview:")
display(df_hhs_quality.head())

print("\nDataset tail:")
display(df_hhs_quality.tail())

# ------------------------------------------------------------
# 6. Current data types
# ------------------------------------------------------------

print("\nCurrent data types after loading structured CSV:")
display(df_hhs_quality.dtypes.to_frame(name="Data Type"))

# ------------------------------------------------------------
# 7. Input readiness summary
# ------------------------------------------------------------

input_readiness_summary = pd.DataFrame({
    "Readiness Check": [
        "Structured File Exists",
        "Rows Loaded",
        "Columns Loaded",
        "Expected Row Count",
        "Expected Column Count",
        "Date Column Present",
        "HHS Care Column Present"
    ],
    "Result": [
        file_exists,
        df_hhs_quality.shape[0],
        df_hhs_quality.shape[1],
        720,
        6,
        "Date" in df_hhs_quality.columns,
        "Children in HHS Care" in df_hhs_quality.columns
    ]
})

print("\nInput readiness summary:")
display(input_readiness_summary)

Step 2A — Load Structured Dataset and Confirm Input Readiness
Project paths configured successfully.
Project root: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children
Structured input file: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\cleaned\01_structured_uac_data.csv

Structured input file exists: True

Structured dataset loaded successfully.
Dataset shape: 720 rows × 6 columns

Column names:
1. Date
2. Children apprehended and placed in CBP custody*
3. Children in CBP custody
4. Children transferred out of CBP custody
5. Children in HHS Care
6. Children discharged from HHS Care

Dataset preview:


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33,53,34,"6,566",436
1,2023-01-22,32,49,39,"7,122",227
2,2023-01-23,32,50,39,"7,280",181
3,2023-01-24,47,42,47,"7,433",175
4,2023-01-25,20,22,41,"7,538",180



Dataset tail:


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
715,2025-12-15,11,42,9,"2,470",7
716,2025-12-16,8,54,15,"2,468",9
717,2025-12-17,7,31,11,"2,481",10
718,2025-12-18,11,50,6,"2,472",16
719,2025-12-21,6,18,11,"2,484",14



Current data types after loading structured CSV:


,Data Type
Date,object
Children apprehended and placed in CBP custody*,int64
Children in CBP custody,int64
Children transferred out of CBP custody,int64
Children in HHS Care,object
Children discharged from HHS Care,int64



Input readiness summary:


,Readiness Check,Result
0,Structured File Exists,True
1,Rows Loaded,720
2,Columns Loaded,6
3,Expected Row Count,720
4,Expected Column Count,6
5,Date Column Present,True
6,HHS Care Column Present,True


## Step 2B — Data Type Validation and Safe Conversion

This step validates and converts the required data types for the structured HHS dataset.

The goal of this step is to:
- Convert the Date column into datetime format
- Clean comma-formatted numeric values in Children in HHS Care
- Convert all care-load and flow columns into numeric integer format
- Check conversion errors
- Confirm final expected data types

No logical constraint validation, anomaly flagging, KPI calculation, or visualization is performed in this step.

In [2]:
# ============================================================
# Step 2B — Data Type Validation and Safe Conversion
# Project: System Capacity & Care Load Analytics for Unaccompanied Children
# ============================================================

print("Step 2B — Data Type Validation and Safe Conversion")
print("=" * 75)

# ------------------------------------------------------------
# 1. Create a working copy for quality validation
# ------------------------------------------------------------

df_hhs_validated = df_hhs_quality.copy()

print("Working validation copy created successfully.")
print(f"Working dataset shape: {df_hhs_validated.shape[0]:,} rows × {df_hhs_validated.shape[1]:,} columns")

# ------------------------------------------------------------
# 2. Define expected columns
# ------------------------------------------------------------

date_column = "Date"

count_columns = [
    "Children apprehended and placed in CBP custody*",
    "Children in CBP custody",
    "Children transferred out of CBP custody",
    "Children in HHS Care",
    "Children discharged from HHS Care"
]

print("\nExpected date column:")
print(date_column)

print("\nExpected count columns:")
for idx, col in enumerate(count_columns, start=1):
    print(f"{idx}. {col}")

# ------------------------------------------------------------
# 3. Date conversion validation
# ------------------------------------------------------------

date_parse_test = pd.to_datetime(df_hhs_validated[date_column], errors="coerce")
date_conversion_errors = date_parse_test.isna().sum()

print(f"\nDate conversion errors: {date_conversion_errors:,}")

if date_conversion_errors > 0:
    print("\nRows with Date conversion errors:")
    display(df_hhs_validated[date_parse_test.isna()])
else:
    print("All Date values converted successfully.")

df_hhs_validated[date_column] = date_parse_test

# ------------------------------------------------------------
# 4. Numeric conversion validation for count columns
# ------------------------------------------------------------

numeric_conversion_summary = []

for col in count_columns:
    original_values = df_hhs_validated[col].copy()

    # Convert comma-formatted values safely
    cleaned_values = (
        original_values
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    converted_values = pd.to_numeric(cleaned_values, errors="coerce")

    conversion_errors = converted_values.isna().sum()

    numeric_conversion_summary.append({
        "Column Name": col,
        "Original Data Type": original_values.dtype,
        "Conversion Errors": conversion_errors,
        "Converted Data Type Before Int": converted_values.dtype
    })

    if conversion_errors > 0:
        print(f"\nRows with numeric conversion errors in column: {col}")
        display(df_hhs_validated[converted_values.isna()][[date_column, col]])

    df_hhs_validated[col] = converted_values.astype("int64")

numeric_conversion_summary_df = pd.DataFrame(numeric_conversion_summary)

print("\nNumeric conversion summary:")
display(numeric_conversion_summary_df)

# ------------------------------------------------------------
# 5. Final data type check
# ------------------------------------------------------------

print("\nFinal data types after safe conversion:")
display(df_hhs_validated.dtypes.to_frame(name="Data Type"))

# ------------------------------------------------------------
# 6. Final conversion readiness summary
# ------------------------------------------------------------

final_dtype_summary = pd.DataFrame({
    "Validation Check": [
        "Date Conversion Errors",
        "Total Numeric Columns Checked",
        "Total Numeric Conversion Errors",
        "Date Final Type",
        "All Count Columns Integer Type",
        "Rows Preserved",
        "Columns Preserved"
    ],
    "Result": [
        date_conversion_errors,
        len(count_columns),
        numeric_conversion_summary_df["Conversion Errors"].sum(),
        str(df_hhs_validated[date_column].dtype),
        all(str(df_hhs_validated[col].dtype) == "int64" for col in count_columns),
        df_hhs_validated.shape[0],
        df_hhs_validated.shape[1]
    ]
})

print("\nFinal conversion readiness summary:")
display(final_dtype_summary)

# ------------------------------------------------------------
# 7. Preview after safe conversion
# ------------------------------------------------------------

print("\nPreview after safe conversion:")
display(df_hhs_validated.head())

print("\nTail after safe conversion:")
display(df_hhs_validated.tail())

Step 2B — Data Type Validation and Safe Conversion
Working validation copy created successfully.
Working dataset shape: 720 rows × 6 columns

Expected date column:
Date

Expected count columns:
1. Children apprehended and placed in CBP custody*
2. Children in CBP custody
3. Children transferred out of CBP custody
4. Children in HHS Care
5. Children discharged from HHS Care

Date conversion errors: 0
All Date values converted successfully.

Numeric conversion summary:


,Column Name,Original Data Type,Conversion Errors,Converted Data Type Before Int
0,Children apprehended and placed in CBP custody*,int64,0,int64
1,Children in CBP custody,int64,0,int64
2,Children transferred out of CBP custody,int64,0,int64
3,Children in HHS Care,object,0,int64
4,Children discharged from HHS Care,int64,0,int64



Final data types after safe conversion:


,Data Type
Date,datetime64[ns]
Children apprehended and placed in CBP custody*,int64
Children in CBP custody,int64
Children transferred out of CBP custody,int64
Children in HHS Care,int64
Children discharged from HHS Care,int64



Final conversion readiness summary:


,Validation Check,Result
0,Date Conversion Errors,0
1,Total Numeric Columns Checked,5
2,Total Numeric Conversion Errors,0
3,Date Final Type,datetime64[ns]
4,All Count Columns Integer Type,True
5,Rows Preserved,720
6,Columns Preserved,6



Preview after safe conversion:


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33,53,34,6566,436
1,2023-01-22,32,49,39,7122,227
2,2023-01-23,32,50,39,7280,181
3,2023-01-24,47,42,47,7433,175
4,2023-01-25,20,22,41,7538,180



Tail after safe conversion:


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
715,2025-12-15,11,42,9,2470,7
716,2025-12-16,8,54,15,2468,9
717,2025-12-17,7,31,11,2481,10
718,2025-12-18,11,50,6,2472,16
719,2025-12-21,6,18,11,2484,14


## Step 2C — Missing Values, Duplicate Dates, and Negative Values Validation

This step validates the cleaned data types before logical constraint checks.

The goal of this step is to:
- Check missing values after safe conversion
- Check duplicate full rows
- Check duplicate reporting dates
- Check negative values in all count columns
- Create a quality validation summary

No logical constraint validation, anomaly flagging, KPI calculation, or visualization is performed in this step.

In [3]:
# ============================================================
# Step 2C — Missing Values, Duplicate Dates, and Negative Values Validation
# Project: System Capacity & Care Load Analytics for Unaccompanied Children
# ============================================================

print("Step 2C — Missing Values, Duplicate Dates, and Negative Values Validation")
print("=" * 85)

# ------------------------------------------------------------
# 1. Confirm working dataset shape
# ------------------------------------------------------------

print("Working dataset for quality validation:")
print(f"Rows: {df_hhs_validated.shape[0]:,}")
print(f"Columns: {df_hhs_validated.shape[1]:,}")

# ------------------------------------------------------------
# 2. Missing values check after safe conversion
# ------------------------------------------------------------

missing_values_after_conversion = pd.DataFrame({
    "Column Name": df_hhs_validated.columns,
    "Missing Values": df_hhs_validated.isna().sum().values,
    "Missing Percentage": (df_hhs_validated.isna().mean().values * 100).round(2)
})

print("\nMissing values after safe conversion:")
display(missing_values_after_conversion)

columns_with_missing_values = (missing_values_after_conversion["Missing Values"] > 0).sum()

# ------------------------------------------------------------
# 3. Duplicate full rows check
# ------------------------------------------------------------

duplicate_full_rows_after_conversion = df_hhs_validated.duplicated().sum()

print(f"\nDuplicate full rows after safe conversion: {duplicate_full_rows_after_conversion:,}")

if duplicate_full_rows_after_conversion > 0:
    print("\nDuplicate full row records:")
    display(df_hhs_validated[df_hhs_validated.duplicated(keep=False)])
else:
    print("No duplicate full rows found after safe conversion.")

# ------------------------------------------------------------
# 4. Duplicate reporting dates check
# ------------------------------------------------------------

duplicate_reporting_dates_after_conversion = df_hhs_validated["Date"].duplicated().sum()

print(f"\nDuplicate reporting dates after safe conversion: {duplicate_reporting_dates_after_conversion:,}")

if duplicate_reporting_dates_after_conversion > 0:
    print("\nDuplicate reporting date records:")
    display(
        df_hhs_validated[df_hhs_validated["Date"].duplicated(keep=False)]
        .sort_values("Date")
    )
else:
    print("No duplicate reporting dates found after safe conversion.")

# ------------------------------------------------------------
# 5. Negative values validation in count columns
# ------------------------------------------------------------

negative_values_summary = []

for col in count_columns:
    negative_count = (df_hhs_validated[col] < 0).sum()
    min_value = df_hhs_validated[col].min()
    max_value = df_hhs_validated[col].max()

    negative_values_summary.append({
        "Column Name": col,
        "Negative Values": negative_count,
        "Minimum Value": min_value,
        "Maximum Value": max_value
    })

negative_values_summary_df = pd.DataFrame(negative_values_summary)

print("\nNegative values validation summary:")
display(negative_values_summary_df)

total_negative_values = negative_values_summary_df["Negative Values"].sum()

if total_negative_values > 0:
    print("\nRows containing negative values:")
    negative_condition = False

    for col in count_columns:
        negative_condition = negative_condition | (df_hhs_validated[col] < 0)

    display(df_hhs_validated[negative_condition])
else:
    print("No negative values found in count columns.")

# ------------------------------------------------------------
# 6. Quality validation summary
# ------------------------------------------------------------

quality_validation_summary = pd.DataFrame({
    "Validation Check": [
        "Rows Checked",
        "Columns Checked",
        "Columns With Missing Values",
        "Total Missing Values",
        "Duplicate Full Rows",
        "Duplicate Reporting Dates",
        "Total Negative Values",
        "Date Column Type",
        "All Count Columns Integer Type"
    ],
    "Result": [
        df_hhs_validated.shape[0],
        df_hhs_validated.shape[1],
        columns_with_missing_values,
        int(missing_values_after_conversion["Missing Values"].sum()),
        int(duplicate_full_rows_after_conversion),
        int(duplicate_reporting_dates_after_conversion),
        int(total_negative_values),
        str(df_hhs_validated["Date"].dtype),
        all(str(df_hhs_validated[col].dtype) == "int64" for col in count_columns)
    ]
})

print("\nQuality validation summary:")
display(quality_validation_summary)

Step 2C — Missing Values, Duplicate Dates, and Negative Values Validation
Working dataset for quality validation:
Rows: 720
Columns: 6

Missing values after safe conversion:


,Column Name,Missing Values,Missing Percentage
0,Date,0,0.0
1,Children apprehended and placed in CBP custody*,0,0.0
2,Children in CBP custody,0,0.0
3,Children transferred out of CBP custody,0,0.0
4,Children in HHS Care,0,0.0
5,Children discharged from HHS Care,0,0.0



Duplicate full rows after safe conversion: 0
No duplicate full rows found after safe conversion.

Duplicate reporting dates after safe conversion: 0
No duplicate reporting dates found after safe conversion.

Negative values validation summary:


,Column Name,Negative Values,Minimum Value,Maximum Value
0,Children apprehended and placed in CBP custody*,0,0,333
1,Children in CBP custody,0,7,531
2,Children transferred out of CBP custody,0,0,440
3,Children in HHS Care,0,1972,11516
4,Children discharged from HHS Care,0,0,505


No negative values found in count columns.

Quality validation summary:


,Validation Check,Result
0,Rows Checked,720
1,Columns Checked,6
2,Columns With Missing Values,0
3,Total Missing Values,0
4,Duplicate Full Rows,0
5,Duplicate Reporting Dates,0
6,Total Negative Values,0
7,Date Column Type,datetime64[ns]
8,All Count Columns Integer Type,True


## Step 2D — Logical Constraint Validation

This step validates the logical rules defined in the project guidelines.

The goal of this step is to check:
- Whether children transferred out of CBP custody are less than or equal to children in CBP custody
- Whether children discharged from HHS Care are less than or equal to children in HHS Care
- Create transparent anomaly flags for records that violate these rules

No rows are deleted, modified, or corrected in this step. Anomalies are flagged only for transparency.

In [4]:
# ============================================================
# Step 2D — Logical Constraint Validation
# Project: System Capacity & Care Load Analytics for Unaccompanied Children
# ============================================================

print("Step 2D — Logical Constraint Validation")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define logical constraint columns
# ------------------------------------------------------------

cbp_custody_col = "Children in CBP custody"
transfer_col = "Children transferred out of CBP custody"

hhs_care_col = "Children in HHS Care"
discharge_col = "Children discharged from HHS Care"

print("Logical constraints from project guidelines:")
print("1. Children transferred out of CBP custody <= Children in CBP custody")
print("2. Children discharged from HHS Care <= Children in HHS Care")

# ------------------------------------------------------------
# 2. Create anomaly flags
# ------------------------------------------------------------

df_hhs_validated["transfer_exceeds_cbp_custody_flag"] = (
    df_hhs_validated[transfer_col] > df_hhs_validated[cbp_custody_col]
)

df_hhs_validated["discharge_exceeds_hhs_care_flag"] = (
    df_hhs_validated[discharge_col] > df_hhs_validated[hhs_care_col]
)

df_hhs_validated["logical_constraint_anomaly_flag"] = (
    df_hhs_validated["transfer_exceeds_cbp_custody_flag"]
    | df_hhs_validated["discharge_exceeds_hhs_care_flag"]
)

print("\nLogical anomaly flags created successfully.")

# ------------------------------------------------------------
# 3. Count logical constraint violations
# ------------------------------------------------------------

transfer_constraint_violations = df_hhs_validated["transfer_exceeds_cbp_custody_flag"].sum()
discharge_constraint_violations = df_hhs_validated["discharge_exceeds_hhs_care_flag"].sum()
total_logical_anomaly_rows = df_hhs_validated["logical_constraint_anomaly_flag"].sum()

print(f"\nTransfer > CBP custody violations: {transfer_constraint_violations:,}")
print(f"Discharge > HHS care violations: {discharge_constraint_violations:,}")
print(f"Total logical anomaly rows: {total_logical_anomaly_rows:,}")

# ------------------------------------------------------------
# 4. Display transfer constraint anomaly records
# ------------------------------------------------------------

transfer_anomaly_records = df_hhs_validated[
    df_hhs_validated["transfer_exceeds_cbp_custody_flag"]
].copy()

if transfer_constraint_violations > 0:
    print("\nSample transfer constraint anomaly records:")
    display(
        transfer_anomaly_records[
            [
                "Date",
                cbp_custody_col,
                transfer_col,
                "transfer_exceeds_cbp_custody_flag"
            ]
        ].head(10)
    )
else:
    print("\nNo transfer constraint anomalies found.")

# ------------------------------------------------------------
# 5. Display discharge constraint anomaly records
# ------------------------------------------------------------

discharge_anomaly_records = df_hhs_validated[
    df_hhs_validated["discharge_exceeds_hhs_care_flag"]
].copy()

if discharge_constraint_violations > 0:
    print("\nSample discharge constraint anomaly records:")
    display(
        discharge_anomaly_records[
            [
                "Date",
                hhs_care_col,
                discharge_col,
                "discharge_exceeds_hhs_care_flag"
            ]
        ].head(10)
    )
else:
    print("\nNo discharge constraint anomalies found.")

# ------------------------------------------------------------
# 6. Create logical validation summary
# ------------------------------------------------------------

logical_validation_summary = pd.DataFrame({
    "Logical Validation Check": [
        "Rows Checked",
        "Transfer Constraint Rule",
        "Transfer > CBP Custody Violations",
        "Discharge Constraint Rule",
        "Discharge > HHS Care Violations",
        "Total Logical Anomaly Rows",
        "Rows Deleted or Modified"
    ],
    "Result": [
        df_hhs_validated.shape[0],
        "Transfers <= CBP custody",
        int(transfer_constraint_violations),
        "Discharges <= HHS care",
        int(discharge_constraint_violations),
        int(total_logical_anomaly_rows),
        0
    ]
})

print("\nLogical validation summary:")
display(logical_validation_summary)

# ------------------------------------------------------------
# 7. Create combined anomaly report preview
# ------------------------------------------------------------

logical_anomaly_report = df_hhs_validated[
    df_hhs_validated["logical_constraint_anomaly_flag"]
].copy()

if total_logical_anomaly_rows > 0:
    print("\nCombined logical anomaly report preview:")
    display(
        logical_anomaly_report[
            [
                "Date",
                cbp_custody_col,
                transfer_col,
                hhs_care_col,
                discharge_col,
                "transfer_exceeds_cbp_custody_flag",
                "discharge_exceeds_hhs_care_flag",
                "logical_constraint_anomaly_flag"
            ]
        ].head(10)
    )
else:
    print("\nNo logical anomaly records found.")

Step 2D — Logical Constraint Validation
Logical constraints from project guidelines:
1. Children transferred out of CBP custody <= Children in CBP custody
2. Children discharged from HHS Care <= Children in HHS Care

Logical anomaly flags created successfully.

Transfer > CBP custody violations: 86
Discharge > HHS care violations: 0
Total logical anomaly rows: 86

Sample transfer constraint anomaly records:


,Date,Children in CBP custody,Children transferred out of CBP custody,transfer_exceeds_cbp_custody_flag
3,2023-01-24,42,47,True
4,2023-01-25,22,41,True
9,2023-02-02,13,23,True
22,2023-02-22,215,230,True
23,2023-02-23,162,178,True
27,2023-03-01,158,193,True
47,2023-04-03,139,168,True
63,2023-04-25,230,242,True
70,2023-05-04,253,286,True
79,2023-05-17,162,193,True



No discharge constraint anomalies found.

Logical validation summary:


,Logical Validation Check,Result
0,Rows Checked,720
1,Transfer Constraint Rule,Transfers <= CBP custody
2,Transfer > CBP Custody Violations,86
3,Discharge Constraint Rule,Discharges <= HHS care
4,Discharge > HHS Care Violations,0
5,Total Logical Anomaly Rows,86
6,Rows Deleted or Modified,0



Combined logical anomaly report preview:


,Date,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,transfer_exceeds_cbp_custody_flag,discharge_exceeds_hhs_care_flag,logical_constraint_anomaly_flag
3,2023-01-24,42,47,7433,175,True,False,True
4,2023-01-25,22,41,7538,180,True,False,True
9,2023-02-02,13,23,7879,298,True,False,True
22,2023-02-22,215,230,7978,232,True,False,True
23,2023-02-23,162,178,7914,386,True,False,True
27,2023-03-01,158,193,7930,307,True,False,True
47,2023-04-03,139,168,8054,284,True,False,True
63,2023-04-25,230,242,8017,202,True,False,True
70,2023-05-04,253,286,8790,332,True,False,True
79,2023-05-17,162,193,8569,307,True,False,True


## Step 2E — Save Validated Dataset and Data Quality Outputs

This step completes Notebook 02 by saving the final validated data quality outputs.

The goal of this step is to:
- Save the validated dataset with logical anomaly flags
- Save the final data quality summary
- Save the logical anomaly report
- Confirm output file locations

No KPI calculation, feature engineering, visualization, or business insight generation is performed in this step.

In [5]:
# ============================================================
# Step 2E — Save Validated Dataset and Data Quality Outputs
# Project: System Capacity & Care Load Analytics for Unaccompanied Children
# ============================================================

print("Step 2E — Save Validated Dataset and Data Quality Outputs")
print("=" * 80)

# ------------------------------------------------------------
# 1. Ensure output directories exist
# ------------------------------------------------------------

DATA_CLEANED_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Output directories verified successfully.")
print(f"Cleaned data directory: {DATA_CLEANED_DIR}")
print(f"Processed data directory: {DATA_PROCESSED_DIR}")

# ------------------------------------------------------------
# 2. Create final data quality summary
# ------------------------------------------------------------

final_data_quality_summary = pd.DataFrame({
    "Quality Check": [
        "Rows Checked",
        "Columns Before Flags",
        "Columns After Flags",
        "Missing Values",
        "Duplicate Full Rows",
        "Duplicate Reporting Dates",
        "Negative Values",
        "Date Column Type",
        "All Count Columns Integer Type",
        "Transfer > CBP Custody Violations",
        "Discharge > HHS Care Violations",
        "Total Logical Anomaly Rows",
        "Rows Deleted or Modified"
    ],
    "Result": [
        df_hhs_validated.shape[0],
        6,
        df_hhs_validated.shape[1],
        int(missing_values_after_conversion["Missing Values"].sum()),
        int(duplicate_full_rows_after_conversion),
        int(duplicate_reporting_dates_after_conversion),
        int(total_negative_values),
        str(df_hhs_validated["Date"].dtype),
        all(str(df_hhs_validated[col].dtype) == "int64" for col in count_columns),
        int(transfer_constraint_violations),
        int(discharge_constraint_violations),
        int(total_logical_anomaly_rows),
        0
    ]
})

print("\nFinal data quality summary:")
display(final_data_quality_summary)

# ------------------------------------------------------------
# 3. Create final logical anomaly report
# ------------------------------------------------------------

logical_anomaly_report = df_hhs_validated[
    df_hhs_validated["logical_constraint_anomaly_flag"]
].copy()

print(f"\nLogical anomaly report rows: {logical_anomaly_report.shape[0]:,}")
print(f"Logical anomaly report columns: {logical_anomaly_report.shape[1]:,}")

if logical_anomaly_report.shape[0] > 0:
    print("\nLogical anomaly report preview:")
    display(logical_anomaly_report.head(10))
else:
    print("\nNo logical anomalies found. Empty anomaly report will still be saved for transparency.")

# ------------------------------------------------------------
# 4. Define output file paths
# ------------------------------------------------------------

validated_data_output_path = DATA_CLEANED_DIR / "02_validated_uac_data.csv"
quality_summary_output_path = DATA_PROCESSED_DIR / "02_data_quality_summary.csv"
logical_anomaly_report_output_path = DATA_PROCESSED_DIR / "02_logical_anomaly_report.csv"

# ------------------------------------------------------------
# 5. Save validated dataset
# ------------------------------------------------------------

df_hhs_validated.to_csv(validated_data_output_path, index=False)

print("\nValidated dataset saved successfully.")
print(f"File: {validated_data_output_path}")

# ------------------------------------------------------------
# 6. Save final data quality summary
# ------------------------------------------------------------

final_data_quality_summary.to_csv(quality_summary_output_path, index=False)

print("\nFinal data quality summary saved successfully.")
print(f"File: {quality_summary_output_path}")

# ------------------------------------------------------------
# 7. Save logical anomaly report
# ------------------------------------------------------------

logical_anomaly_report.to_csv(logical_anomaly_report_output_path, index=False)

print("\nLogical anomaly report saved successfully.")
print(f"File: {logical_anomaly_report_output_path}")

# ------------------------------------------------------------
# 8. Saved outputs summary
# ------------------------------------------------------------

saved_quality_outputs_summary = pd.DataFrame({
    "Output Name": [
        "Validated UAC Dataset",
        "Data Quality Summary",
        "Logical Anomaly Report"
    ],
    "File Path": [
        str(validated_data_output_path),
        str(quality_summary_output_path),
        str(logical_anomaly_report_output_path)
    ],
    "Expected Purpose": [
        "Validated dataset with logical anomaly flags for Notebook 03",
        "Final summary of data quality and validation checks",
        "Transparent report of rows violating guideline logical constraints"
    ]
})

print("\nSaved quality outputs summary:")
display(saved_quality_outputs_summary)

Step 2E — Save Validated Dataset and Data Quality Outputs
Output directories verified successfully.
Cleaned data directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\cleaned
Processed data directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\processed

Final data quality summary:


,Quality Check,Result
0,Rows Checked,720
1,Columns Before Flags,6
2,Columns After Flags,9
3,Missing Values,0
4,Duplicate Full Rows,0
5,Duplicate Reporting Dates,0
6,Negative Values,0
7,Date Column Type,datetime64[ns]
8,All Count Columns Integer Type,True
9,Transfer > CBP Custody Violations,86



Logical anomaly report rows: 86
Logical anomaly report columns: 9

Logical anomaly report preview:


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,transfer_exceeds_cbp_custody_flag,discharge_exceeds_hhs_care_flag,logical_constraint_anomaly_flag
3,2023-01-24,47,42,47,7433,175,True,False,True
4,2023-01-25,20,22,41,7538,180,True,False,True
9,2023-02-02,15,13,23,7879,298,True,False,True
22,2023-02-22,107,215,230,7978,232,True,False,True
23,2023-02-23,101,162,178,7914,386,True,False,True
27,2023-03-01,84,158,193,7930,307,True,False,True
47,2023-04-03,118,139,168,8054,284,True,False,True
63,2023-04-25,81,230,242,8017,202,True,False,True
70,2023-05-04,153,253,286,8790,332,True,False,True
79,2023-05-17,57,162,193,8569,307,True,False,True



Validated dataset saved successfully.
File: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\cleaned\02_validated_uac_data.csv

Final data quality summary saved successfully.
File: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\processed\02_data_quality_summary.csv

Logical anomaly report saved successfully.
File: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care Load Analytics for Unaccompanied Children\data\processed\02_logical_anomaly_report.csv

Saved quality outputs summary:


,Output Name,File Path,Expected Purpose
0,Validated UAC Dataset,C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care ...,Validated dataset with logical anomaly flags for Notebook 03
1,Data Quality Summary,C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care ...,Final summary of data quality and validation checks
2,Logical Anomaly Report,C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\System Capacity & Care ...,Transparent report of rows violating guideline logical constraints
